# Test control: inctal vs. Preictal
Tried and true - Ween

Compare ictal vs. preictal to have a positive control. 
Same process as try2_feat_eng but the aim is to create a df with only windows coming from ictal and preictal.

In [1]:
import os
import numpy as np
import pandas as pd
from datetime import datetime, timezone
from pathlib import Path
from typing import Optional

## 1. Load DATA

In [16]:
files = Path("/home/tperezsanchez/FoundationModel_EEG_Dissertation/EEG_data_vis/results/XB47Y_28032026Normalized/")
base_files = sorted(files.glob("*full.npz"))

print(f"Found {len(base_files)} .npz files")

#for f in base_files:
    #print(f"  {f.name}") 
    # print all the files names

Found 183 .npz files


This code builds a structured metadata table from a collection of EEG `.npz` files while ensuring robustness to inconsistent formats. For each file, it extracts key information such as sampling frequency (`fs`), timestamps (`T0`, `TF`), channel names, seizure onsets, and signal shape without fully loading the data into memory. A helper function (`parse_timestamp`) standardizes timestamps by handling multiple possible formats (floats, strings, or arrays) and converting them into Unix time. The code also detects whether recordings were normalized, captures basic signal dimensions (`n_channels`, `n_samples`), and logs any loading errors to avoid pipeline failure. Overall, this step creates a clean, unified metadata structure that can be used for downstream analysis while safeguarding against malformed or heterogeneous input files.
Created a function to correctly process the datetime data. Also there was a problem with T0/TF in files that contained seizures. Repetead values that were generated when each onset was stored in key. Repetead TO/TF per seizure event. 
So in order to solve this a "unique" selection mechanism was applied. 


In [3]:
rows = []
from datetime import datetime, timezone
import numpy as np


def parse_timestamp(val):
    """Accept Unix float, datetime string, or repeated timestamp arrays."""

    if isinstance(val, np.ndarray):
        flat = val.ravel()
        cleaned = [str(x).strip() for x in flat if str(x).strip() != ""]

        if len(cleaned) == 0:
            return None

        unique_vals = list(dict.fromkeys(cleaned))

        if len(unique_vals) == 1:
            val = unique_vals[0]
        else:
            raise ValueError(f"Timestamp array has multiple different values: {unique_vals}")

    try:
        return float(val)
    except (ValueError, TypeError):
        pass

    val = str(val).strip()
    dt = datetime.strptime(val, "%Y-%m-%d %H:%M:%S.%f")
    return dt.replace(tzinfo=timezone.utc).timestamp()
    
for base_NPZ_path in base_files:
    meta = {"file_name": base_NPZ_path.name, "file_path": str(base_NPZ_path.resolve()), "load_error": None}

    try:
        with np.load(base_NPZ_path, allow_pickle=True) as data:
            keys = set(data.files)
            # store data from key into meta df, convert them into float or str first
            
            meta["fs"]         = float(data["fs"]) if "fs" in keys else None
            meta["source_file"]= str(data["source_file"]) if "source_file" in keys else None
            meta["T0"] = parse_timestamp(data["T0"]) if "T0" in keys else None
            meta["TF"] = parse_timestamp(data["TF"]) if "TF" in keys else None
            meta["is_normalized"] = "mu" in keys and "sigma" in keys

            meta["channel_names"]  = list(data["channel_names"]) if "channel_names" in keys else []
            meta["seizure_onsets"] = list(data["seizure_onsets"]) if "seizure_onsets" in keys else []

            # Read X shape only — avoids loading the full array into memory
            if "X" in keys:
                shape = data["X"].shape
                meta["n_channels"] = int(shape[0]) if len(shape) == 2 else None
                meta["n_samples"]  = int(shape[1]) if len(shape) == 2 else None
            else:
                meta["n_channels"] = None
                meta["n_samples"]  = None

    except Exception as e:
        meta["load_error"] = str(e)
        print(f"Failed to load {base_NPZ_path.name}: {e}")

    rows.append(meta)

print(f"\nLoaded metadata from {len(rows)} file(s)")


Loaded metadata from 183 file(s)


## 2. Temporal sanity check
This code performs a data integrity check on the EEG metadata for each recording. It converts the start (`T0`) and end (`TF`) timestamps into human-readable datetime objects, computes the recording duration in seconds, and verifies consistency between the duration derived from timestamps (`TF - T0`) and the duration inferred from the signal itself (`n_samples / fs`). If the difference between these two estimates exceeds a tolerance (1 second), the recording is flagged as potentially inconsistent. Finally, it reports how many files show a duration mismatch, helping identify corrupted or misaligned data before further processing.


In [4]:
from datetime import datetime

for meta in rows:
    T0 = meta.get("T0")
    TF = meta.get("TF") 

    meta["start_time"] = datetime.fromtimestamp(T0) if T0 is not None else None
    meta["end_time"]   = datetime.fromtimestamp(TF) if TF is not None else None
    meta["duration_s"] = round(TF - T0, 3) if (T0 is not None and TF is not None) else None

    # Cross-check: does n_samples / fs match TF - T0?
    fs, n = meta.get("fs"), meta.get("n_samples")
    if fs and n and meta["duration_s"] is not None:
        expected = round(n / fs, 3)
        meta["duration_check_ok"] = abs(expected - meta["duration_s"]) < 1.0
    else:
        meta["duration_check_ok"] = None

bad = [m for m in rows if m["duration_check_ok"] is False]
print(len(bad), "files with duration mismatch")

0 files with duration mismatch


## 3. General Dataframe for everyfile 

In [5]:
COLUMNS = [
    "file_name", "file_path",
    "start_time", "end_time", "duration_s",
    "fs", "n_channels", "n_samples",
    "channel_names", "seizure_onsets",
    "is_normalized", "source_file",
    "duration_check_ok", "load_error",
]

df = pd.DataFrame(rows, columns=COLUMNS)
df = df.sort_values("start_time", na_position="last").reset_index(drop=True)

print(df.shape)
df.head()

(183, 14)


,file_name,file_path,start_time,end_time,duration_s,fs,n_channels,n_samples,channel_names,seizure_onsets,is_normalized,source_file,duration_check_ok,load_error
0,XB47Y_35_preproc_full.npz,/home/tperezsanchez/FoundationModel_EEG_Disser...,2019-10-29 09:31:04,2019-10-29 10:01:00.730450,1796.730,207.031055,2,371979,"[EEG SQ_D-SQ_C, EEG SQ_P-SQ_C]",[nan],True,['XB47Y_35.mat'],True,None
1,XB47Y_37_preproc_full.npz,/home/tperezsanchez/FoundationModel_EEG_Disser...,2019-10-29 19:54:13,2019-10-30 01:54:12.759550,21599.760,207.031055,2,4471821,"[EEG SQ_D-SQ_C, EEG SQ_P-SQ_C]",[nan],True,['XB47Y_37.mat'],True,None
2,XB47Y_38_preproc_full.npz,/home/tperezsanchez/FoundationModel_EEG_Disser...,2019-10-30 01:54:13,2019-10-30 04:58:50.338150,11077.338,207.031055,2,2293353,"[EEG SQ_D-SQ_C, EEG SQ_P-SQ_C]",[nan],True,['XB47Y_38.mat'],True,None
3,XB47Y_98_preproc_full.npz,/home/tperezsanchez/FoundationModel_EEG_Disser...,2019-10-30 07:47:44,2019-10-30 13:47:44.759400,21600.759,207.031055,2,4472028,"[EEG SQ_D-SQ_C, EEG SQ_P-SQ_C]",[nan],True,['XB47Y_98.mat'],True,None
4,XB47Y_99_preproc_full.npz,/home/tperezsanchez/FoundationModel_EEG_Disser...,2019-10-30 13:47:44,2019-10-30 19:47:44.759400,21600.759,207.031055,2,4472028,"[EEG SQ_D-SQ_C, EEG SQ_P-SQ_C]",[nan],True,['XB47Y_99.mat'],True,None


In [6]:
print("── Sanity Report ──────────────────────────────────────")
print(f"  Total recordings   : {len(df)}")
print(f"  Load errors        : {df['load_error'].notna().sum()}")
print(f"  Missing T0/TF      : {df['start_time'].isna().sum()}")
print(f"  Duration check ✗   : {(df['duration_check_ok'] == False).sum()}")
print(f"  Missing fs         : {df['fs'].isna().sum()}")
print(f"  Normalized files   : {df['is_normalized'].sum()}")

# Check for overlapping recordings
valid = df.dropna(subset=["start_time", "end_time"]).sort_values("start_time")
overlaps = 0
for i in range(len(valid) - 1):
    if valid.iloc[i]["end_time"] > valid.iloc[i + 1]["start_time"]:
        overlaps += 1
print(f"  Overlapping pairs  : {overlaps}")
print("───────────────────────────────────────────────────────")

── Sanity Report ──────────────────────────────────────
  Total recordings   : 183
  Load errors        : 0
  Missing T0/TF      : 0
  Duration check ✗   : 0
  Missing fs         : 0
  Normalized files   : 183
  Overlapping pairs  : 75
───────────────────────────────────────────────────────


### 4. Clean onsets: solution to problems with "nan"

This function cleans and standardizes the seizure_onsets column to ensure a consistent format across all entries. It converts any input into a list of valid onset timestamps by removing missing values (NaN) when the input is already a list or array, returning an empty list if the value is missing, and wrapping single values into a list. The result is a new column where each row contains a clean list of seizure onset times, which simplifies downstream processing and avoids errors during iteration or labeling steps.

In [7]:
def clean_onsets(x):
    if isinstance(x, (list, np.ndarray)):
        return [i for i in x if not pd.isna(i)]
    elif pd.isna(x):
        return []
    else:
        return [x]

df["seizure_onsets_clean"] = df["seizure_onsets"].apply(clean_onsets)
#df[["seizure_onsets", "seizure_onsets_clean"]].head(20)

## 5. Windowing

In [8]:
# Create an empty list that will store one dictionary per EEG window
rows_windows = []

# Loop through each row of the original dataframe 'df'
# Each row represents one EEG file / recording
for idx, row in df.iterrows():
     # Get the full path of the current .npz file
    file_path = row["file_path"]
    data = np.load(file_path, allow_pickle=True)
    # Extract the EEG signal array
    # Expected shape: (channels, samples)
    X = data["X"]
    # Read the sampling frequency from the dataframe
    fs = row["fs"]

    # Define the window length in samples
    # Example: 10 seconds * 207 Hz ≈ 2070 samples
    window_size = int(10 * fs)
    # Get the total number of samples in the recording
    # X.shape[1] = number of columns = number of time samples
    N = X.shape[1]
    
    # Compute how many full 10-second windows fit into the recording
    # '//' means integer division, so incomplete last windows are ignored IMPORTANT!!1
    n_windows = N // window_size

    # Take seizure onsets from the recording-level dataframe
    seizure_onsets = row["seizure_onsets_clean"]
    # Loop through each window index
    for w in range(n_windows):
        # Compute the start sample of the current window
        start = w * window_size
        # Compute the end sample of the current window
        end = start + window_size
        
        # Slice the EEG signal to extract only this window
        # Shape will be (channels, window_size)
        window = X[:, start:end]
        # Store metadata about this window as one dictionary
        rows_windows.append({
            "file_name": row["file_name"],
            "window_id": w,
            "start_sample": start,
            "end_sample": end,
            "fs": fs,
            "n_channels": X.shape[0],
            "seizure_onsets": seizure_onsets
        })

In [9]:
# Convert the list of dictionaries into a new dataframe
# Each row now represents one 10-second window
df_windows = pd.DataFrame(rows_windows)

print(df_windows.head())
print(df_windows.shape) # rows vs. columns

                   file_name  window_id  start_sample  end_sample          fs  \
0  XB47Y_35_preproc_full.npz          0             0        2070  207.031055   
1  XB47Y_35_preproc_full.npz          1          2070        4140  207.031055   
2  XB47Y_35_preproc_full.npz          2          4140        6210  207.031055   
3  XB47Y_35_preproc_full.npz          3          6210        8280  207.031055   
4  XB47Y_35_preproc_full.npz          4          8280       10350  207.031055   

   n_channels seizure_onsets  
0           2             []  
1           2             []  
2           2             []  
3           2             []  
4           2             []  
(291113, 7)


In [24]:
df_windows[df_windows["seizure_onsets"].apply(lambda x: not pd.isna(x).all() if isinstance(x, (list, np.ndarray)) else not pd.isna(x))].head(10)

,file_name,window_id,start_sample,end_sample,fs,n_channels,seizure_onsets
15409,XB47Y_41_preproc_full.npz,0,0,2070,207.031055,2,[2019-10-31 23:25:08.153000]
15410,XB47Y_41_preproc_full.npz,1,2070,4140,207.031055,2,[2019-10-31 23:25:08.153000]
15411,XB47Y_41_preproc_full.npz,2,4140,6210,207.031055,2,[2019-10-31 23:25:08.153000]
15412,XB47Y_41_preproc_full.npz,3,6210,8280,207.031055,2,[2019-10-31 23:25:08.153000]
15413,XB47Y_41_preproc_full.npz,4,8280,10350,207.031055,2,[2019-10-31 23:25:08.153000]
15414,XB47Y_41_preproc_full.npz,5,10350,12420,207.031055,2,[2019-10-31 23:25:08.153000]
15415,XB47Y_41_preproc_full.npz,6,12420,14490,207.031055,2,[2019-10-31 23:25:08.153000]
15416,XB47Y_41_preproc_full.npz,7,14490,16560,207.031055,2,[2019-10-31 23:25:08.153000]
15417,XB47Y_41_preproc_full.npz,8,16560,18630,207.031055,2,[2019-10-31 23:25:08.153000]
15418,XB47Y_41_preproc_full.npz,9,18630,20700,207.031055,2,[2019-10-31 23:25:08.153000]


This code filters the DataFrame to keep only rows where seizure_onsets contains at least one valid (non-NaN) value, and then displays the first 10 such rows.

## 6. labelling
* Computes the real start and end timestamp of each EEG window using the recording start time, sample indices, and sampling frequency.
* Creates two explicit label columns:

  * `class_label` for numeric encoding
  * `label_name` for human-readable class names
* Defines the **preictal interval** as the period from **10 minutes to 5 minutes before seizure onset**.
* Defines the **seizure interval** as the period from **seizure onset to 5 minutes after onset**.
* Checks whether each window overlaps with any seizure-related interval.
* Uses **partial overlap** as sufficient for labeling, so windows do not need to be fully contained within the interval.
* Assigns **seizure** priority over **preictal** if a window could match both conditions.
* Excludes windows in the gap between **5 minutes before onset and the onset itself**.
* Excludes all windows that are neither preictal nor seizure-related.
* Produces a final filtered DataFrame that keeps:

  * all original columns
  * computed window timestamps
  * the two new label columns
  * only windows labeled as **preictal** or **seizure**

**CHEQUED**


In [10]:
from datetime import timedelta
import pandas as pd
import numpy as np

# -------------------------------
# Helper: clean seizure_onsets
# -------------------------------
def clean_onsets(x):
    if isinstance(x, (list, np.ndarray, pd.Series)):
        return [i for i in x if not pd.isna(i)]
    elif pd.isna(x) or x is None:
        return []
    else:
        return [x]

# -------------------------------
# Copy dataframe to avoid overwriting original
# -------------------------------
df_windows_labeled = df_windows.copy()

# Create new columns
df_windows_labeled["window_start_time"] = pd.NaT
df_windows_labeled["window_end_time"] = pd.NaT
df_windows_labeled["class_label"] = np.nan
df_windows_labeled["label_name"] = pd.NA

# -------------------------------
# Main labeling loop
# -------------------------------
for idx, row in df_windows_labeled.iterrows():
    
    # Match recording-level metadata
    rec = df[df["file_name"] == row["file_name"]].iloc[0]

    # Recording start time
    recording_start = pd.to_datetime(rec["start_time"])
    fs = row["fs"]
    
    # Convert sample indices to seconds
    start_sec = row["start_sample"] / fs
    end_sec = row["end_sample"] / fs

    # Compute real datetime of each window
    window_start_time = recording_start + pd.Timedelta(seconds=start_sec)
    window_end_time = recording_start + pd.Timedelta(seconds=end_sec)

    # Save as datetime first
    df_windows_labeled.at[idx, "window_start_time"] = window_start_time
    df_windows_labeled.at[idx, "window_end_time"] = window_end_time
    
    # Default = excluded
    assigned_label = np.nan
    assigned_name = pd.NA

    # Clean seizure_onsets
    seizure_onsets = clean_onsets(row["seizure_onsets"])
    
    # Loop through every onset
    for onset in seizure_onsets:
        onset = pd.to_datetime(onset)
        
        # Define intervals
        preictal_start = onset - pd.Timedelta(minutes=10)
        preictal_end   = onset - pd.Timedelta(minutes=5)
        seizure_start  = onset
        seizure_end    = onset + pd.Timedelta(minutes=5)
        
        # Overlap function
        def overlaps(a_start, a_end, b_start, b_end):
            return (a_start < b_end) and (a_end > b_start)

        # 1) Seizure has priority
        if overlaps(window_start_time, window_end_time, seizure_start, seizure_end):
            assigned_label = 1
            assigned_name = "seizure"
            break
        
        # 2) Preictal only if not seizure
        elif overlaps(window_start_time, window_end_time, preictal_start, preictal_end):
            if pd.isna(assigned_label) or assigned_label != 1:
                assigned_label = 0
                assigned_name = "preictal"
    
    df_windows_labeled.at[idx, "class_label"] = assigned_label
    df_windows_labeled.at[idx, "label_name"] = assigned_name

# -------------------------------
# Keep only preictal and seizure windows
# -------------------------------
df_final = df_windows_labeled[df_windows_labeled["class_label"].isin([0, 1])].copy()

# Make class_label integer
df_final["class_label"] = df_final["class_label"].astype(int)

# -------------------------------
# Convert datetime columns to visible string format
# -------------------------------
df_final["window_start_time"] = pd.to_datetime(df_final["window_start_time"]).dt.strftime("%Y-%m-%d %H:%M:%S")
df_final["window_end_time"] = pd.to_datetime(df_final["window_end_time"]).dt.strftime("%Y-%m-%d %H:%M:%S")

# -------------------------------
# Quick sanity check
# -------------------------------
print(df_final[[
    "file_name",
    "window_id",
    "window_start_time",
    "window_end_time",
    "seizure_onsets",
    "class_label",
    "label_name"
]].head(30))

print("\nClass counts:")
print(df_final["label_name"].value_counts())

                       file_name  window_id    window_start_time  \
16663  XB47Y_41_preproc_full.npz       1254  2019-10-31 23:15:03   
16664  XB47Y_41_preproc_full.npz       1255  2019-10-31 23:15:13   
16665  XB47Y_41_preproc_full.npz       1256  2019-10-31 23:15:23   
16666  XB47Y_41_preproc_full.npz       1257  2019-10-31 23:15:33   
16667  XB47Y_41_preproc_full.npz       1258  2019-10-31 23:15:43   
16668  XB47Y_41_preproc_full.npz       1259  2019-10-31 23:15:53   
16669  XB47Y_41_preproc_full.npz       1260  2019-10-31 23:16:03   
16670  XB47Y_41_preproc_full.npz       1261  2019-10-31 23:16:13   
16671  XB47Y_41_preproc_full.npz       1262  2019-10-31 23:16:23   
16672  XB47Y_41_preproc_full.npz       1263  2019-10-31 23:16:33   
16673  XB47Y_41_preproc_full.npz       1264  2019-10-31 23:16:43   
16674  XB47Y_41_preproc_full.npz       1265  2019-10-31 23:16:53   
16675  XB47Y_41_preproc_full.npz       1266  2019-10-31 23:17:03   
16676  XB47Y_41_preproc_full.npz       1267  201

In [11]:
df[df["file_name"] == "XB47Y_41_preproc_full.npz"]
# just manually checking that this makes sense

,file_name,file_path,start_time,end_time,duration_s,fs,n_channels,n_samples,channel_names,seizure_onsets,is_normalized,source_file,duration_check_ok,load_error,seizure_onsets_clean
9,XB47Y_41_preproc_full.npz,/home/tperezsanchez/FoundationModel_EEG_Disser...,2019-10-31 19:46:05,2019-11-01 01:46:05.759400,21600.759,207.031055,2,4472028,"[EEG SQ_D-SQ_C, EEG SQ_P-SQ_C]",[2019-10-31 23:25:08.153000],True,['XB47Y_41.mat'],True,None,[2019-10-31 23:25:08.153000]


## 7. Feature extraction


In [12]:
# TIME-DOMAIN FEATURES EXTRACTION
from scipy.stats import skew, kurtosis
def extract_time_features(window, channel_names=None):
    """
    Extract per-channel time-domain features from one EEG window.

    Parameters
    ----------
    window : np.ndarray
        Shape (C, window_samples)
    channel_names : list or None
        Optional list of channel names

    Returns
    -------
    features : dict
        Flat dictionary with one feature per channel
    """
    
    n_channels = window.shape[0]
    features = {}
    
    for ch in range(n_channels):
        x = window[ch, :]
        
        # Choose channel label
        if channel_names is not None and ch < len(channel_names):
            ch_name = str(channel_names[ch])
        else:
            ch_name = f"ch{ch+1}"
        
        # Optional: make names safer for DataFrame columns
        ch_name = ch_name.replace(" ", "_")
        
        features[f"mean_{ch_name}"] = np.mean(x)
        features[f"std_{ch_name}"] = np.std(x)
        features[f"var_{ch_name}"] = np.var(x)
        features[f"rms_{ch_name}"] = np.sqrt(np.mean(x**2))
        features[f"ptp_{ch_name}"] = np.ptp(x)   # max - min
        features[f"line_length_{ch_name}"] = np.sum(np.abs(np.diff(x)))
        features[f"skew_{ch_name}"] = skew(x, bias=False)
        features[f"kurtosis_{ch_name}"] = kurtosis(x, bias=False)
        if np.std(x) < 1e-6:
            features[f"skew_{ch_name}"] = np.nan
            features[f"kurtosis_{ch_name}"] = np.nan
        else:
            features[f"skew_{ch_name}"] = skew(x, bias=False)
            features[f"kurtosis_{ch_name}"] = kurtosis(x, bias=False)
    
    return features

In [13]:
import numpy as np
from scipy.signal import welch

def extract_frequency_features(window, fs, channel_names=None):
    """
    Extract per-channel frequency-domain features from one EEG window.

    Parameters
    ----------
    window : np.ndarray
        Shape (C, window_samples)
    fs : float
        Sampling frequency in Hz
    channel_names : list or None
        Optional list of channel names

    Returns
    -------
    features : dict
        Flat dictionary with one feature per channel
    """

    # Standard EEG bands (Hz)
    bands = {
        "delta": (0.5, 4.0),
        "theta": (4.0, 8.0),
        "alpha": (8.0, 13.0),
        "beta":  (13.0, 30.0),
        "gamma": (30.0, 40.0),
    }

    n_channels = window.shape[0]
    features = {}

    for ch in range(n_channels):
        x = window[ch, :]

        # Choose channel label
        if channel_names is not None and ch < len(channel_names):
            ch_name = str(channel_names[ch])
        else:
            ch_name = f"ch{ch+1}"

        # Make names safer for DataFrame columns
        ch_name = ch_name.replace(" ", "_")

        # Welch PSD
        # For 10 s windows this is usually fine
        nperseg = min(len(x), int(fs * 2))  # 2-second segments
        noverlap = nperseg // 2

        freqs, psd = welch(
            x,
            fs=fs,
            window="hann",
            nperseg=nperseg,
            noverlap=noverlap,
            detrend="constant",
            scaling="density"
        )

        # Band powers
        for band_name, (f_low, f_high) in bands.items():
            mask = (freqs >= f_low) & (freqs < f_high)
                    
            if np.any(mask):
                band_power = np.trapezoid(psd[mask], freqs[mask])
            else:
                band_power = np.nan

            features[f"{band_name}_power_{ch_name}"] = band_power

        # Peak frequency in the analyzed EEG range
        eeg_mask = (freqs >= 0.5) & (freqs <= 40.0)
        if np.any(eeg_mask):
            peak_idx = np.argmax(psd[eeg_mask])
            peak_freq = freqs[eeg_mask][peak_idx]
        else:
            peak_freq = np.nan

        features[f"peak_frequency_{ch_name}"] = peak_freq

    return features

## 8. Main loop
